In [ ]:
from snowflake.snowpark.context import get_active_session
import subprocess
import sys

session = get_active_session()
session.file.get("@demo_db.additional_packages.python_packages/openpyxl-3.1.5-py2.py3-none-any.whl", "/tmp")

subprocess.check_call([sys.executable, "-m", "pip", "install", "/tmp/openpyxl-3.1.5-py2.py3-none-any.whl", "--no-deps", "--quiet"])
# python -m pip install /tmp/openpyxl-3.1.5-py2.py3-none-any.whl --no-deps --quiet (Equivalent of running this in terminal)

import openpyxl
print(f"openpyxl {openpyxl.__version__} installed successfully")

# 01 Load Excel Files

* Author: Jeremiah Hansen
* Last Updated: 2/27/2026

This notebook will load data into the `LOCATION` and `ORDER_DETAIL` tables from Excel files.

This currently does not use Snowpark File Access as it doesn't yet work in Notebooks. So for now we copy the file locally first.

In [ ]:
# Import python packages
import sys
import logging

# Set up the logger
logger_name = 'demo_logger'
logger = logging.getLogger(logger_name)
logger.setLevel(logging.INFO)

# Set default values for debugging
notebook_name = '01_load_excel_files.ipynb'
database_name = 'DEMO_DB'
schema_name = 'DEV_SCHEMA'
role_name = 'DEMO_ROLE'

# Override values with passed notebook arguments
if sys.argv[0].endswith('.ipynb'):
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--database-name', type=str)
    parser.add_argument('--schema-name', type=str)
    parser.add_argument('--role-name', type=str)
    args, args_unknown = parser.parse_known_args()

    notebook_name = parser.prog  # same as argv[0]
    database_name = args.database_name or database_name
    schema_name = args.schema_name or schema_name
    role_name = args.role_name or role_name

# Get a Snowpark session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Set the default database and schema for the following cells
session.use_schema(f"{database_name}.{schema_name}")

# Set the role
# Needed when running EXECUTE NOTEBOOK PROJECT directly (since it ignores session context and uses the user's default role)
session.use_role(role_name)

# Get details about the current state
current_state_df = session.sql(f"""
        SELECT OBJECT_CONSTRUCT(
            'current_user', CURRENT_USER(),
            'current_role', CURRENT_ROLE(),
            'current_secondary_roles', PARSE_JSON(CURRENT_SECONDARY_ROLES()),
            'current_database', CURRENT_DATABASE(),
            'current_schema', CURRENT_SCHEMA()
        )::STRING AS session_context;
    """).collect()

logger.info(f"Begin executing notebook {notebook_name}", extra = {'logger_name': logger_name})
logger.info(f"Using parameters database: {database_name}, schema: {schema_name}, role: {role_name}", extra = {'logger_name': logger_name})
logger.info(f"Using session context {current_state_df[0]['SESSION_CONTEXT']}", extra = {'logger_name': logger_name})

In [ ]:
!pip install openpyxl

In [ ]:
import zipfile
import xml.etree.ElementTree as ET
from io import BytesIO
import pandas as pd
from datetime import datetime, timedelta

def read_xlsx_no_openpyxl(file_bytes, sheet_name):
    NS = '{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
    with zipfile.ZipFile(BytesIO(file_bytes)) as z:
        shared = []
        if 'xl/sharedStrings.xml' in z.namelist():
            tree = ET.parse(z.open('xl/sharedStrings.xml'))
            for si in tree.getroot().findall(f'{NS}si'):
                t_el = si.find(f'{NS}t')
                if t_el is not None and t_el.text is not None:
                    shared.append(t_el.text)
                else:
                    parts = [r.find(f'{NS}t').text or '' for r in si.findall(f'.//{NS}r') if r.find(f'{NS}t') is not None]
                    shared.append(''.join(parts))

        wb_tree = ET.parse(z.open('xl/workbook.xml'))
        sheets = wb_tree.getroot().findall(f'.//{NS}sheet')
        sheet_idx = None
        for i, s in enumerate(sheets):
            if s.get('name') == sheet_name:
                sheet_idx = i + 1
                break
        if sheet_idx is None:
            raise ValueError(f"Sheet '{sheet_name}' not found. Available: {[s.get('name') for s in sheets]}")

        sheet_tree = ET.parse(z.open(f'xl/worksheets/sheet{sheet_idx}.xml'))
        rows_data = []
        for row in sheet_tree.getroot().findall(f'.//{NS}row'):
            row_cells = {}
            for cell in row.findall(f'{NS}c'):
                ref = cell.get('r')
                col_letter = ''.join(c for c in ref if c.isalpha())
                col_idx = 0
                for c in col_letter:
                    col_idx = col_idx * 26 + (ord(c) - ord('A') + 1)
                col_idx -= 1
                v_el = cell.find(f'{NS}v')
                val = v_el.text if v_el is not None else None
                cell_type = cell.get('t')
                if cell_type == 's' and val is not None:
                    val = shared[int(val)]
                elif val is not None and cell_type != 's':
                    try:
                        val = float(val)
                        if val == int(val):
                            val = int(val)
                    except (ValueError, OverflowError):
                        pass
                row_cells[col_idx] = val
            rows_data.append(row_cells)

        if not rows_data:
            return pd.DataFrame()
        max_col = max(max(r.keys()) for r in rows_data if r) + 1
        header = {i: rows_data[0].get(i, f'col_{i}') for i in range(max_col)}
        data_rows = [{header[i]: r.get(i) for i in range(max_col)} for r in rows_data[1:]]
        return pd.DataFrame(data_rows)

print('Excel reader ready (no openpyxl needed)')

In [ ]:
%%sql -r dataframe_1
-- Temporary solution to load in the metadata, this should be replaced with a directy query to a directory table (or a metadata table)
SELECT '@INTEGRATIONS.FROSTBYTE_RAW_STAGE/intro/order_detail.xlsx' AS STAGE_FILE_PATH, 'order_detail' AS WORKSHEET_NAME, 'ORDER_DETAIL' AS TARGET_TABLE
UNION
SELECT '@INTEGRATIONS.FROSTBYTE_RAW_STAGE/intro/location.xlsx', 'location', 'LOCATION';

## Create a function to load Excel worksheet to table

Create a reusable function to load an Excel worksheet to a table in Snowflake.

Note: Until we can use scoped URLs in Notebooks, via the `BUILD_SCOPED_FILE_URL()` function, we need to temporarily copy the file to a temp stage and then process from there.

In [ ]:
from snowflake.snowpark.files import SnowflakeFile
from openpyxl import load_workbook
import pandas as pd

# 1. Create a temp internal stage (once at the start)
session.sql("CREATE TEMP STAGE IF NOT EXISTS temp_excel_stage").collect()

def load_excel_worksheet_to_table(session, external_path, worksheet_name, target_table):
    """Load an Excel worksheet by copying to internal stage first"""
    
    # Extract filename from path
    filename = external_path.split('/')[-1]
    
    # 2. Copy file from external to internal stage
    session.sql(f"""
        COPY FILES INTO @temp_excel_stage
        FROM {external_path}
    """).collect()
    
    # 3. Now SnowflakeFile.open() works on internal stage
    with SnowflakeFile.open(f'@temp_excel_stage/{filename}', 'rb') as f:
        workbook = load_workbook(f)
        sheet = workbook[worksheet_name]
        
    # Convert to DataFrame
    data = sheet.values
    columns = next(data)
    df = pd.DataFrame(data, columns=columns)
    
    # Write to Snowflake table
    snowpark_df = session.create_dataframe(df)
    snowpark_df.write.mode("overwrite").save_as_table(target_table)
    
    logger.info(f"Loaded {len(df)} rows from '{worksheet_name}' to {target_table}", extra = {'logger_name': logger_name})

## Process all Excel worksheets

Loop through each Excel worksheet to process and call our `load_excel_worksheet_to_table_local()` function.

In [ ]:
# Process each file from the sql_get_spreadsheets cell above
files_to_load = dataframe_1
for index, excel_file in files_to_load.iterrows():
    print(f"Processing Excel file {excel_file['STAGE_FILE_PATH']}")
    load_excel_worksheet_to_table(session, excel_file['STAGE_FILE_PATH'], excel_file['WORKSHEET_NAME'], excel_file['TARGET_TABLE'])

logger.info(f"Finish executing notebook {notebook_name}", extra = {'logger_name': logger_name})

In [ ]:
%%sql -r dataframe_4
create table location_test as (
select * from location where 1 = 2
)
;

create table order_detail_test as (
select * from order_detail where 1 = 2
)
;

### Debugging

In [ ]:
%%sql -r dataframe_2
--DESCRIBE TABLE LOCATION;
--SELECT * FROM LOCATION;
--SHOW TABLES;